In [0]:
# Read bronze customers table

bronze_customers_df = spark.table("bronze.bronze_customers")

bronze_customers_df.show(5)

In [0]:
from pyspark.sql.functions import *

# -----------------------------
# CLEAN CUSTOMER DATA
# -----------------------------

silver_customers_df = bronze_customers_df \
    .dropDuplicates(["customer_id"]) \
    .withColumn(
        "customer_name",
        initcap(col("customer_name"))
    ) \
    .withColumn(
        "email",
        lower(col("email"))
    ) \
    .withColumn(
        "city",
        initcap(col("city"))
    ) \
    .withColumn(
        "country",
        initcap(col("country"))
    ) \
    .withColumn(
        "signup_date",
        to_date(col("signup_date"))
    ) \
    .withColumn(
        "signup_year",
        year(col("signup_date"))
    )

# Remove null customer IDs
silver_customers_df = silver_customers_df.filter(
    col("customer_id").isNotNull()
)

print("Customer cleaning completed!")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

In [0]:
silver_customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.silver_customers")

print("Silver customers table created!")

In [0]:
spark.sql("SELECT * FROM silver.silver_customers LIMIT 10").show()